# AI Personal Tutor (RAG + Output Parser)

# Document loading and cleaning

In [ ]:
!pip install -q langchain langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:

pdf_path = "/kaggle/input/datasets/hazemwaleed5/computernetwork/Computer Networks 5th Edition (A. Tanenbaum).pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()
print("Pages loaded:", len(documents))

In [ ]:
def clean_documents(documents):
    cleaned_documents = []

    for doc in documents:
        text = doc.page_content.strip()

        if not text:
            continue

        if text.lower() == "this page intentionally left blank":
            continue

        doc.page_content = text

        cleaned_documents.append(doc)

    return cleaned_documents

In [ ]:
documents = clean_documents(documents)
print("Pages after cleaning:", len(documents))

# Chunking

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

In [ ]:
chunks = text_splitter.split_documents(documents)
print("Chunks:", len(chunks))

# Embeddings + Vector Database

In [ ]:
!pip install -q sentence-transformers langchain-huggingface faiss-cpu

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)

In [ ]:
vectorstore = FAISS.from_documents(
    chunks,
    embedding
)
print("Vector store ready")

# Build Rag

In [ ]:
!pip install -q transformers accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Open model (no token needed). To use Mistral again:
# model_name = "mistralai/Mistral-7B-Instruct-v0.2"   (needs HF token)
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

In [ ]:
def generate_text(prompt, max_new_tokens=700, num_return_sequences=1):
    messages = [{"role": "user", "content": prompt}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.4,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    result = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return result.strip()

# Tutor Explanation

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
explain_prompt = PromptTemplate(
    input_variables=["topic", "context"],
    template="""
You are an AI Personal Tutor for university students.

Explain the topic using ONLY information explicitly stated in the context.

Topic:
{topic}

Context:
{context}

Give the explanation in this format:

Definition:
A simple definition based directly on the context.

Explanation:
A detailed explanation suitable for a university student.

Key Points:
- Important point 1
- Important point 2
- Important point 3

Example:
Give an example only if the context explicitly provides one. Otherwise write: No example is given in the material.

STRICT RULES:
- Use ONLY the provided context.
- Do NOT use outside knowledge.
- Do NOT make assumptions or inferences.
- Do NOT combine information from different concepts unless the context explicitly connects them.
- Every statement must be directly supported by the context.
- If a fact is not explicitly stated in the context, do not mention it.
"""
)

In [ ]:
def explain_topic(topic, vectorstore):

    results = vectorstore.similarity_search(
        topic,
        k=3
    )

    context = "\n\n".join(
        doc.page_content for doc in results
    )
    prompt = explain_prompt.format(
        topic=topic,
        context=context
    )

    answer = generate_text(
        prompt,
        max_new_tokens=400
    )

    return answer

In [ ]:
question = "Wide Area Networks"

answer = explain_topic(
    question,
    vectorstore
)

print(answer)

# Parsing quiz 

In [ ]:
from pydantic import BaseModel
from typing import Dict, List


class Question(BaseModel):
    question: str
    options: Dict[str, str]
    correct_answer: str


class Quiz(BaseModel):
    questions: List[Question]

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=Quiz)

In [ ]:
format_instructions = parser.get_format_instructions()

# Question Generation

In [ ]:
quiz_prompt = PromptTemplate(
    input_variables=["topic", "context", "format_instructions"],
    template="""
You are an AI Personal Tutor for university students.

Generate a quiz about the requested topic using ONLY the provided context.

Topic:
{topic}

Context:
{context}

Generate exactly 3 multiple-choice questions.

Rules:
- Each question must have exactly 4 options.
- Options must be A, B, C, and D.
- Each question must have exactly one correct answer.
- Each question must be based ONLY on the provided context.
- Do NOT use outside knowledge.
- Do NOT make assumptions or inferences.
- Each question should test a different concept from the context.
- Do NOT create questions about information that is not explicitly stated.
- Make sure the correct answer is clearly supported by the context.

Return the result using exactly these format instructions:

{format_instructions}
"""
)

In [ ]:
def generate_quiz(topic, vectorstore):

    results = vectorstore.similarity_search(
        topic,
        k=3
    )

    context = "\n\n".join(
        doc.page_content for doc in results
    )

    prompt = quiz_prompt.format(
        topic=topic,
        context=context,
        format_instructions=format_instructions
    )

    quiz = generate_text(
        prompt,
        max_new_tokens=700
    )

    return quiz

In [ ]:
q = "Wide Area Networks"
quiz = generate_quiz(
    q,
    vectorstore
)

print(quiz)

In [ ]:
parsed_quiz = parser.parse(quiz)
parsed_quiz

# Answer Evaluation

In [ ]:
student_answers = ["C", "B", "A"]

In [ ]:
def evaluate_answers(quiz, student_answers):

    score = 0
    results = []

    questions = quiz.questions

    for i, question in enumerate(questions):

        # Get the student's answer safely
        if i < len(student_answers):
            student_answer = str(
                student_answers[i]
            ).strip().upper()
        else:
            student_answer = ""

        # Get the original correct answer
        correct_answer = str(
            question.correct_answer
        ).strip().upper()

        # Compare option letters
        is_correct = (
            student_answer == correct_answer
            and student_answer in question.options
        )

        if is_correct:
            score += 1

        results.append({
            "question": question.question,
            "student_answer": student_answer,
            "correct_answer": correct_answer,
            "is_correct": is_correct
        })

    return {
        "score": score,
        "total": len(questions),
        "results": results
    }

In [ ]:
evaluation = evaluate_answers(
    parsed_quiz,
    student_answers
)

print(evaluation)

In [ ]:
evaluation_prompt = PromptTemplate(
    input_variables=[
        "score",
        "total",
        "mistakes_with_context"
    ],
    template="""
You are an AI Personal Tutor for university students.

Your task is to explain mistakes that have ALREADY been identified
and verified by Python.

Student Score:
{score}/{total}

Verified Incorrect Questions:
{mistakes_with_context}

STRICT RULES:

1. The score is calculated by Python and is final.
2. NEVER recalculate or change the score.
3. Only explain the mistakes supplied above.
4. Do NOT add, remove, or renumber mistakes.
5. Preserve every question number exactly.
6. Preserve the student's answer letter and answer text exactly.
7. Preserve the correct answer letter and answer text exactly.
8. Do NOT decide whether an answer is correct or incorrect.
9. Do NOT replace the correct answer with another option.
10. Explain the difference between the two answers using ONLY
    the context supplied for that specific question.
11. Do NOT use outside knowledge.
12. Do NOT invent facts or explanations.
13. If the context is insufficient, write:
    "The provided context is insufficient to explain this mistake."
14. Recommendations must be based only on the supplied context.
15. Do NOT mention questions that are not included in the mistakes.
16. Do NOT output a different score from the one provided.

Use exactly this format:

Question [number]:
Student Answer: [exact student answer letter and text]
Correct Answer: [exact correct answer letter and text]
Explanation: [brief explanation based only on the corresponding context]
What to Review: [brief recommendation based only on the corresponding context]

Do not add any additional sections.
"""
)

In [ ]:
def generate_feedback(evaluation, quiz, vectorstore):

    mistakes = []

    # Collect only incorrect answers
    for i, result in enumerate(evaluation["results"]):

        if not result["is_correct"]:

            question = quiz.questions[i]

            student_answer = str(result["student_answer"]).strip().upper()
            correct_answer = str(result["correct_answer"]).strip().upper()

            mistakes.append({
                "question_number": i + 1,
                "question": question.question,
                "student_answer": student_answer,
                "student_answer_text": question.options.get(
                    student_answer, "Invalid answer"
                ),
                "correct_answer": correct_answer,
                "correct_answer_text": question.options.get(
                    correct_answer, "Invalid answer"
                )
            })

    overall = (
        f"Overall Feedback:\n"
        f"The student answered {evaluation['score']} out of "
        f"{evaluation['total']} questions correctly.\n\n"
    )

    # If all answers are correct
    if not mistakes:
        return overall + "Excellent work!"

    # Retrieve context for each mistake
    mistakes_with_context = []

    for mistake in mistakes:

        results = vectorstore.similarity_search(
            mistake["question"],
            k=3
        )

        context = "\n\n".join(
            doc.page_content for doc in results
        )

        mistakes_with_context.append({
            "mistake": mistake,
            "context": context
        })

    score = evaluation["score"]
    total = evaluation["total"]

    prompt = evaluation_prompt.format(
        score=score,
        total=total,
        mistakes_with_context=mistakes_with_context
    )

    feedback = generate_text(
        prompt,
        max_new_tokens=500
    )

    return overall + "Mistakes:\n\n" + feedback

In [ ]:
feedback = generate_feedback(
    evaluation,
    parsed_quiz,
    vectorstore
)

print(feedback)

# web

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
import shutil

In [ ]:
app = FastAPI()

current_quiz = None
current_topic = None
current_explanation = None

In [ ]:
class GenerateRequest(BaseModel):
    topic: str


class QuizSubmission(BaseModel):
    answers: list[str]

In [ ]:
@app.post("/upload")
def upload_pdf(file: UploadFile = File(...)):

    global vectorstore

    pdf_path = "/kaggle/working/uploaded.pdf"

    with open(pdf_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    loader = PyPDFLoader(pdf_path)

    documents = loader.load()

    documents = clean_documents(documents)

    chunks = text_splitter.split_documents(documents)

    vectorstore = FAISS.from_documents(
        chunks,
        embedding
    )

    return {
        "message": "PDF uploaded and processed successfully"
    }

In [ ]:
@app.post("/generate")
def generate_content(request: GenerateRequest):

    global current_quiz
    global current_topic
    global current_explanation

    if vectorstore is None:
        return {
            "error": "Please upload a PDF first"
        }

    current_topic = request.topic

    current_explanation = explain_topic(
        request.topic,
        vectorstore
    )

    
    current_quiz = None
    for _ in range(3):
        quiz_text = generate_quiz(
            request.topic,
            vectorstore
        )
        try:
            current_quiz = parser.parse(quiz_text)
            break
        except Exception:
            continue

    if current_quiz is None:
        raise HTTPException(500, "Could not generate a valid quiz, try again")

    
    quiz_for_student = [
        {"question": q.question, "options": q.options}
        for q in current_quiz.questions
    ]

    return {
        "topic": current_topic,
        "explanation": current_explanation,
        "quiz": quiz_for_student
    }

In [ ]:
@app.post("/submit-quiz")
def submit_quiz(submission: QuizSubmission):

    if current_quiz is None:
        return {
            "error": "Please generate a quiz first"
        }

    evaluation = evaluate_answers(
        current_quiz,
        submission.answers
    )

    feedback = generate_feedback(
        evaluation,
        current_quiz,
        vectorstore
    )

    return {
        "topic": current_topic,
        "explanation": current_explanation,
        "quiz": current_quiz.model_dump(),
        "student_score": {
            "score": evaluation["score"],
            "total": evaluation["total"]
        },
        "feedback": feedback
    }

In [ ]:
!pip install pyngrok

In [ ]:
import socket
import threading
import time

from pyngrok import ngrok, conf
import uvicorn

In [ ]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port


port = free_port()

# Put your NEW ngrok token here (the old one was shown publicly, revoke it)
conf.get_default().auth_token = "PUT_YOUR_NGROK_TOKEN_HERE"


def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=port
    )


threading.Thread(
    target=run,
    daemon=True
).start()

time.sleep(2)

public_url = ngrok.connect(port).public_url

print("Your public URL:", public_url)
print("Swagger:", public_url + "/docs")